## Catastro Minero de Salta
### Notebook: Análisis longitudinal entre cortes (base: marzo 2026)
### Proyecto: Análisis del catastro minero oficial de la provincia de Salta, Argentina
###Autora: Camila Mopty
#### Fecha: 2026

## 1. Librerías y preparación

In [ ]:
#@title
#LIBRERÍAS

import subprocess
import os
import warnings
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import geopandas as gpd
from shapely.validation import make_valid

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import importlib
for lib in ['numpy','pandas','geopandas','shapely','pyproj','fiona',
            'folium','plotly','mapclassify','sklearn']:
    try:
        print(f"{lib:12s} {importlib.import_module(lib).__version__}")
    except Exception as e:
        print(f"{lib:12s} no instalada ({e})")

In [ ]:
!pip freeze | grep -Ei "^(numpy|pandas|geopandas|shapely|pyproj|fiona|folium|plotly|mapclassify|scikit-learn)==" > requirements.txt

In [ ]:
import os, random
import numpy as np

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
#@title
# Clonación del repositorio

repo_url = "https://github.com/Camilamop/catastro-minero-salta-mining-registry-salta.git"
repo_name = "catastro-minero-salta-mining-registry-salta"

if not os.path.exists(repo_name):
    subprocess.run(["git", "clone", repo_url], check=True)

else:
    print("El repositorio ya existe en el entorno.")

## 2. Carga y limpieza de los cortes

Cortes comparados: marzo 2026 (base), junio 2026 y agosto 2026.

In [ ]:
#@title
# Función de carga y limpieza (misma lógica que el EDA)

CRITICOS = {'LI', 'CU', 'AU'}

def es_critico(valor):
    if pd.isna(valor):
        return False
    tokens = {t.strip().upper() for t in str(valor).split(',')}
    return bool(tokens & CRITICOS)

STEM = "poligonos_adaf55f39328ff45c8c60e94eb13a7a0.shp"

def cargar_corte(carpeta):
    # Leer el shapefile
    g = gpd.read_file(f"catastro-minero-salta-mining-registry-salta/data/raw/{carpeta}/{STEM}",
                      encoding='latin-1')
    g = g.to_crs(epsg=4326)

    mask_invalidas = ~g.geometry.is_valid
    g.loc[mask_invalidas, 'geometry'] = g.loc[mask_invalidas, 'geometry'].apply(make_valid)

    # Transformación: campo 'area' a float
    g['area_ha'] = (
        g['area']
        .str.replace(' ha', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
    )
    g['fecha_inic'] = pd.to_datetime(g['fecha_inic'], errors='coerce', dayfirst=True)

    # Subconjunto con mineral y clasificación de minerales críticos
    g = g[g['mineral'].notna() & (g['mineral'].str.strip() != '')].copy()
    g['minerales_criticos'] = g['mineral'].apply(es_critico)
    return g

In [ ]:
#@title
# Carga de los tres cortes

SNAPS = ['marzo_2026', 'junio_2026', 'agosto_2026']
ETIQUETA = {'marzo_2026': 'Marzo 2026', 'junio_2026': 'Junio 2026', 'agosto_2026': 'Agosto 2026'}

cortes_raw = {s: cargar_corte(s) for s in SNAPS}

for s in SNAPS:
    print(f"{ETIQUETA[s]:14s} filas: {len(cortes_raw[s]):,}")

## 3. Tabla a nivel expediente

La clave de cruce entre cortes es `expediente`. Un mismo expediente puede venir en varios polígonos (multipolígono), así que se agrega a un registro por expediente: superficie sumada, atributos por primer registro y flag crítico por máximo.

In [ ]:
#@title
# Tabla a nivel expediente por corte

def tabla_expediente(g, snapshot):
    g = g[g['expediente'].notna()].copy()
    t = (
        g.groupby('expediente')
        .agg(
            estado=('estado', 'first'),
            municipio=('municipio', 'first'),
            departamen=('departamen', 'first'),
            concesiona=('concesiona', 'first'),
            tipo=('tipo', 'first'),
            minerales_criticos=('minerales_criticos', 'max'),
            area_ha=('area_ha', 'sum'),
            fecha_inic=('fecha_inic', 'min'),
            n_poligonos=('expediente', 'size'),
        )
        .reset_index()
    )
    t['snapshot'] = snapshot
    return t

cortes = {s: tabla_expediente(cortes_raw[s], s) for s in SNAPS}

for s in SNAPS:
    print(f"{ETIQUETA[s]:14s} expedientes: {len(cortes[s]):,}")

## 4. Resumen por corte

In [ ]:
#@title
# Resumen comparativo por corte

def resumen_corte(t):
    crit = t[t['minerales_criticos'] == True]
    return pd.Series({
        'Expedientes': len(t),
        'Expedientes críticos': len(crit),
        'Superficie total (ha)': t['area_ha'].sum(),
        'Superficie crítica (ha)': crit['area_ha'].sum(),
        '% críticos': round(len(crit) / len(t) * 100, 1),
    })

resumen = pd.DataFrame({ETIQUETA[s]: resumen_corte(cortes[s]) for s in SNAPS})
display(resumen)

In [ ]:
#@title
# Gráfico 1: Cantidad de expedientes con y sin minerales críticos por corte

etiquetas = [ETIQUETA[s] for s in SNAPS]
con = [int((cortes[s]['minerales_criticos'] == True).sum()) for s in SNAPS]
sin = [int((cortes[s]['minerales_criticos'] == False).sum()) for s in SNAPS]

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Con minerales críticos (Li, Cu o Au)',
    x=etiquetas, y=con,
    marker_color='#C0392B',
    text=con, textposition='outside'
))

fig.add_trace(go.Bar(
    name='Sin minerales críticos',
    x=etiquetas, y=sin,
    marker_color='#E8834A',
    text=sin, textposition='outside'
))

fig.update_layout(
    barmode='group',
    separators='.,',
    title=dict(
        text='Cantidad de expedientes por corte',
        font=dict(size=15, color='#2C3E50'),
        x=0.05
    ),
    xaxis=dict(title=''),
    yaxis=dict(title='Cantidad de expedientes', showgrid=True,
               gridcolor='#E0E0E0', gridwidth=1, griddash='dot'),
    legend=dict(orientation='h', y=-0.12),
    margin=dict(t=60, b=80, l=60, r=40),
    width=850, height=500,
    plot_bgcolor='white', paper_bgcolor='white'
)

fig.show()

## 5. Altas y bajas entre cortes

Altas: expedientes presentes en el corte más reciente que no estaban en el anterior. Bajas: expedientes del corte anterior que ya no aparecen.

In [ ]:
#@title
# Altas y bajas por transición (base: marzo)

TRANSICIONES = [('marzo_2026', 'junio_2026'), ('junio_2026', 'agosto_2026')]

def altas_bajas(a, b):
    sa, sb = set(a['expediente']), set(b['expediente'])
    altas = sb - sa
    bajas = sa - sb
    b_alt = b[b['expediente'].isin(altas)]
    a_baj = a[a['expediente'].isin(bajas)]
    return {
        'altas_criticas': int(b_alt['minerales_criticos'].sum()),
        'altas_no_criticas': int((~b_alt['minerales_criticos']).sum()),
        'bajas_criticas': int(a_baj['minerales_criticos'].sum()),
        'bajas_no_criticas': int((~a_baj['minerales_criticos']).sum()),
        'permanecen': len(sa & sb),
    }

filas = []
for a, b in TRANSICIONES:
    d = altas_bajas(cortes[a], cortes[b])
    d['transicion'] = f"{ETIQUETA[a]} → {ETIQUETA[b]}"
    filas.append(d)

mov = pd.DataFrame(filas).set_index('transicion')
display(mov)

In [ ]:
#@title
# Gráfico 2: Altas (der.) y bajas (izq.) por transición, según minerales críticos

y = list(mov.index)

fig = go.Figure()

fig.add_trace(go.Bar(name='Altas críticas', y=y, x=mov['altas_criticas'],
                     orientation='h', marker_color='#C0392B',
                     text=mov['altas_criticas'], textposition='outside'))
fig.add_trace(go.Bar(name='Altas no críticas', y=y, x=mov['altas_no_criticas'],
                     orientation='h', marker_color='#E8834A',
                     text=mov['altas_no_criticas'], textposition='outside'))
fig.add_trace(go.Bar(name='Bajas críticas', y=y, x=-mov['bajas_criticas'],
                     orientation='h', marker_color='#7B241C',
                     text=mov['bajas_criticas'], textposition='outside'))
fig.add_trace(go.Bar(name='Bajas no críticas', y=y, x=-mov['bajas_no_criticas'],
                     orientation='h', marker_color='#B9770E',
                     text=mov['bajas_no_criticas'], textposition='outside'))

fig.update_layout(
    barmode='relative',
    separators='.,',
    title=dict(text='Altas y bajas de expedientes por transición',
               font=dict(size=15, color='#2C3E50'), x=0.05),
    xaxis=dict(title='Bajas  ←     0     →  Altas', showgrid=True,
               gridcolor='#E0E0E0', gridwidth=1, griddash='dot', zeroline=True,
               zerolinecolor='#2C3E50'),
    yaxis=dict(title=''),
    legend=dict(orientation='h', y=-0.15),
    margin=dict(t=60, b=90, l=140, r=60),
    width=900, height=450,
    plot_bgcolor='white', paper_bgcolor='white'
)

fig.show()

## 6. Cambios de estado

Sobre los expedientes presentes en marzo y en agosto, se compara el estado inicial y el final.

In [ ]:
#@title
# Matriz de transición de estado (marzo → agosto)

a, b = cortes['marzo_2026'], cortes['agosto_2026']
j = (a[['expediente', 'estado']]
     .merge(b[['expediente', 'estado']], on='expediente', suffixes=('_marzo', '_agosto')))

matriz = pd.crosstab(j['estado_marzo'], j['estado_agosto'])
display(matriz)

cambios = j[j['estado_marzo'] != j['estado_agosto']]
print(f"Expedientes presentes en ambos cortes: {len(j):,}")
print(f"Con cambio de estado: {len(cambios):,}")
display(cambios.groupby(['estado_marzo', 'estado_agosto']).size()
        .reset_index(name='cantidad').sort_values('cantidad', ascending=False))

## 7. Variación de superficie por municipio

Superficie de expedientes con minerales críticos, sumada por municipio en cada corte.

In [ ]:
#@title
# Variación de superficie crítica por municipio

def pivot_superficie(col, solo_criticos=True):
    partes = []
    for s in SNAPS:
        t = cortes[s]
        if solo_criticos:
            t = t[t['minerales_criticos'] == True]
        partes.append(t.groupby(col)['area_ha'].sum().rename(ETIQUETA[s]))
    P = pd.concat(partes, axis=1).fillna(0)
    P['Δ marzo→agosto'] = P[ETIQUETA['agosto_2026']] - P[ETIQUETA['marzo_2026']]
    return P.sort_values('Δ marzo→agosto', ascending=False)

muni = pivot_superficie('municipio')
display(muni.head(12).round(0))

In [ ]:
#@title
# Gráfico 3: Top municipios por variación de superficie crítica (marzo → agosto)

top = muni.head(12).sort_values('Δ marzo→agosto')

fig = go.Figure(go.Bar(
    y=top.index, x=top['Δ marzo→agosto'],
    orientation='h', marker_color='#C0392B',
    text=[f"{v:,.0f} ha".replace(',', '.') for v in top['Δ marzo→agosto']],
    textposition='outside'
))

fig.update_layout(
    separators='.,',
    title=dict(text='Variación de superficie crítica por municipio (marzo → agosto)',
               font=dict(size=15, color='#2C3E50'), x=0.05),
    xaxis=dict(title='Hectáreas', tickformat=',.0f', showgrid=True,
               gridcolor='#E0E0E0', gridwidth=1, griddash='dot'),
    yaxis=dict(title=''),
    margin=dict(t=60, b=60, l=180, r=120),
    width=900, height=550,
    plot_bgcolor='white', paper_bgcolor='white'
)

fig.show()

## 8. Variación de superficie por actor

Superficie crítica acumulada por titular (`concesiona`) en cada corte. Se excluye 'Vacancia Solicitada'. La titularidad todavía no está normalizada.

In [ ]:
#@title
# Variación de superficie crítica por actor

actor = pivot_superficie('concesiona')
actor = actor[~actor.index.isin(['Vacancia Solicitada'])]
display(actor.head(12).round(0))

In [ ]:
#@title
# Gráfico 4: Top actores por variación de superficie crítica (marzo → agosto)

top = actor.head(12).sort_values('Δ marzo→agosto')
etiquetas_y = [a[:45] + ('…' if len(a) > 45 else '') for a in top.index]

fig = go.Figure(go.Bar(
    y=etiquetas_y, x=top['Δ marzo→agosto'],
    orientation='h', marker_color='#4a5d7b',
    text=[f"{v:,.0f} ha".replace(',', '.') for v in top['Δ marzo→agosto']],
    textposition='outside'
))

fig.update_layout(
    separators='.,',
    title=dict(text='Variación de superficie crítica por actor (marzo → agosto)',
               font=dict(size=15, color='#2C3E50'), x=0.05),
    xaxis=dict(title='Hectáreas', tickformat=',.0f', showgrid=True,
               gridcolor='#E0E0E0', gridwidth=1, griddash='dot'),
    yaxis=dict(title=''),
    margin=dict(t=60, b=60, l=320, r=120),
    width=1000, height=550,
    plot_bgcolor='white', paper_bgcolor='white'
)

fig.show()

## 9. Aceleración de minerales críticos

Altas de expedientes con minerales críticos por municipio, acumuladas sobre las dos transiciones. Es el proxy de aceleración de la presión territorial.

In [ ]:
#@title
# Altas críticas por municipio (marzo→junio y junio→agosto)

def altas_criticas_por_municipio(a, b):
    nuevos = set(b['expediente']) - set(a['expediente'])
    bn = b[b['expediente'].isin(nuevos) & (b['minerales_criticos'] == True)]
    return bn.groupby('municipio').size()

acel = pd.concat(
    [altas_criticas_por_municipio(cortes[a], cortes[b]).rename(f"{ETIQUETA[a]}→{ETIQUETA[b]}")
     for a, b in TRANSICIONES],
    axis=1
).fillna(0).astype(int)
acel['Total altas críticas'] = acel.sum(axis=1)
acel = acel.sort_values('Total altas críticas', ascending=False)
display(acel.head(12))

In [ ]:
#@title
# Gráfico 5: Aceleración - altas críticas por municipio

top = acel[acel['Total altas críticas'] > 0].sort_values('Total altas críticas')
cols = [f"{ETIQUETA[a]}→{ETIQUETA[b]}" for a, b in TRANSICIONES]

fig = go.Figure()
colores = ['#C0392B', '#E8834A']
for col, c in zip(cols, colores):
    fig.add_trace(go.Bar(name=col, y=top.index, x=top[col],
                         orientation='h', marker_color=c))

fig.update_layout(
    barmode='stack',
    separators='.,',
    title=dict(text='Altas de expedientes críticos por municipio',
               font=dict(size=15, color='#2C3E50'), x=0.05),
    xaxis=dict(title='Cantidad de altas', showgrid=True,
               gridcolor='#E0E0E0', gridwidth=1, griddash='dot'),
    yaxis=dict(title=''),
    legend=dict(orientation='h', y=-0.15),
    margin=dict(t=60, b=90, l=180, r=60),
    width=900, height=550,
    plot_bgcolor='white', paper_bgcolor='white'
)

fig.show()

In [ ]:
# Ejecutá esta celda AL FINAL del notebook
import time
time.sleep(3)  # le da tiempo a Colab de sincronizar